In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/caf-dataset/CAF_sensors.dbf
/kaggle/input/caf-dataset/Hourly/Hourly/CAF141.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF019.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF209.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF007.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF397.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF237.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF205.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF125.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF139.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF095.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF201.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF215.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF079.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF231.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF275.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF003.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF197.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF351.txt
/kaggle/input/caf-dataset/Hourly/Hourly/

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader CPU-safe

does not r

from sklearn.neighbors import NearestNeighbors


In [3]:
df_wide = pd.read_parquet("/kaggle/input/baseline-artifacts/df_wide.parquet")
import geopandas as gpd

DBF_PATH = "/kaggle/input/caf-dataset/CAF_sensors.dbf"

gdf = gpd.read_file(DBF_PATH)



df_wide = df_wide.sort_index()
sensor_ids = df_wide.columns.tolist()

print(df_wide.shape)
# print(sensor_meta.head())


(66705, 10)


In [4]:
# ---------------------------
# FIX 1: Handle missing data
# ---------------------------
df_filled = df_wide.interpolate(limit_direction="both")
df_filled = df_filled.fillna(df_filled.mean())

assert not df_filled.isna().any().any(), "NaNs still present!"


In [5]:
# ---------------------------
# FIX 2: Normalize
# ---------------------------
X_np = df_filled.values.astype("float32")

mean = X_np.mean()
std  = X_np.std() + 1e-6

X_np = (X_np - mean) / std


In [6]:
X_np = (X_np - mean) / std
X = torch.tensor(X_np, dtype=torch.float32)


In [7]:
import torch

X = torch.tensor(X_np, dtype=torch.float32)

assert not torch.isnan(X).any()
assert torch.isfinite(X).all()


In [8]:
gdf.head()

,Location,Easting,Northing
0,CAF003,493383,5180586
1,CAF007,493511,5180568
2,CAF009,493575,5180573
3,CAF019,493247,5180590
4,CAF031,493628,5180612


In [9]:
sensor_meta = gdf[["Location", "Easting", "Northing"]].copy()
sensor_meta.head()


,Location,Easting,Northing
0,CAF003,493383,5180586
1,CAF007,493511,5180568
2,CAF009,493575,5180573
3,CAF019,493247,5180590
4,CAF031,493628,5180612


In [10]:
coords = (
    sensor_meta
    .set_index("Location")
    .loc[sensor_ids][["Easting", "Northing"]]
    .values
)

k = 4  # number of neighbors

nbrs = NearestNeighbors(n_neighbors=k+1).fit(coords)
_, indices = nbrs.kneighbors(coords)

edge_index = []
for i in range(len(sensor_ids)):
    for j in indices[i][1:]:
        edge_index.append([i, j])

edge_index = torch.tensor(edge_index, dtype=torch.long).t()

print("Edge index shape:", edge_index.shape)


Edge index shape: torch.Size([2, 40])


In [11]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, data_np, window=24):
        self.X = data_np
        self.window = window


    def __len__(self):
        return len(self.X) - self.window

    def __getitem__(self, idx):
        x = self.X[idx:idx+self.window]
        y = self.X[idx+self.window]
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
        )


In [12]:
class GNN_GRU(nn.Module):
    def __init__(self, num_nodes, hidden_dim):
        super().__init__()

        self.gnn = nn.Linear(1, hidden_dim)
        self.gru = nn.GRU(
            input_size=hidden_dim * num_nodes,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.decoder = nn.Linear(hidden_dim, num_nodes)

    def forward(self, x):
        # x: (B, T, N)
        B, T, N = x.shape

        x = x.unsqueeze(-1)          # (B, T, N, 1)
        x = self.gnn(x)              # (B, T, N, H)
        x = x.reshape(B, T, -1)      # (B, T, N*H)

        _, h = self.gru(x)
        h = h.squeeze(0)

        out = self.decoder(h)
        return out


In [13]:
window = 24

dataset = SpatioTemporalDataset(
    data_np=X_np,
    window=window
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    drop_last=True
)


In [14]:
model = GNN_GRU(
    num_nodes=df_wide.shape[1],
    hidden_dim=32
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


In [15]:
print("NaNs in X:", np.isnan(X_np).any())
print("Infs in X:", np.isinf(X_np).any())


NaNs in X: False
Infs in X: False


In [18]:
epochs = 10

for epoch in range(epochs):
    total_loss = 0
    for x, y in loader:
            if torch.isnan(x).any() or torch.isnan(y).any():
                continue
        
            optimizer.zero_grad()
            y_hat = model(x)

            loss = loss_fn(y_hat, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.6f}")


Epoch 1, Loss: 79.928287
Epoch 2, Loss: 15.100850
Epoch 3, Loss: 5.599224
Epoch 4, Loss: 1.534450
Epoch 5, Loss: 0.458228
Epoch 6, Loss: 0.228534
Epoch 7, Loss: 0.159489
Epoch 8, Loss: 0.132057
Epoch 9, Loss: 0.115919
Epoch 10, Loss: 0.106442
